In [ ]:
import streamlit as st

st.set_page_config(
    page_title="CoCare Chatbot",
    page_icon="💬",
    layout="centered"
)

st.title("CoCare Chatbot")
st.write("نظام شات بوت ذكي لخدمات الاتصالات")

# =========================
# Safe imports
# =========================
try:
    from utils.language_utils import detect_language
except Exception:
    def detect_language(text: str) -> str:
        arabic_count = sum(1 for ch in text if "\u0600" <= ch <= "\u06FF")
        english_count = sum(1 for ch in text if ch.isascii() and ch.isalpha())
        return "ar" if arabic_count >= english_count else "en"

try:
    from utils.intent_utils import predict_intent
    intent_ready = True
    intent_error = ""
except Exception as e:
    predict_intent = None
    intent_ready = False
    intent_error = str(e)

try:
    from utils.sentiment_utils import predict_sentiment
    sentiment_ready = True
    sentiment_error = ""
except Exception as e:
    predict_sentiment = None
    sentiment_ready = False
    sentiment_error = str(e)

try:
    from utils.prediction_utils import predict_network_issue
    prediction_ready = True
    prediction_error = ""
except Exception as e:
    predict_network_issue = None
    prediction_ready = False
    prediction_error = str(e)

try:
    from utils.response_utils import build_final_response
    response_ready = True
    response_error = ""
except Exception as e:
    build_final_response = None
    response_ready = False
    response_error = str(e)


# =========================
# Helper functions
# =========================
def fallback_response(intent: str, lang: str) -> str:
    arabic_responses = {
        "greeting": "أهلًا وسهلًا، كيف أقدر أساعدك؟",
        "slow_internet": "واضح أن الإنترنت بطيء. جربي إعادة تشغيل الراوتر ثم أخبريني بالنتيجة.",
        "no_signal": "يبدو أن هناك مشكلة في الإشارة. تأكدي من التغطية وأعيدي تشغيل الجهاز.",
        "payment_issue": "واضح أن هناك مشكلة في الدفع. تأكدي من بيانات الدفع ثم حاولي مرة أخرى.",
        "renew_package": "أقدر أساعدك بتجديد الباقة. ما اسم الباقة التي تريدين تجديدها؟",
        "offer_inquiry": "أقدر أساعدك بمعرفة العروض المتاحة. ما نوع العرض الذي تبحثين عنه؟",
        "technical_support": "تمام، اشرحي لي المشكلة الفنية بالتفصيل.",
        "network_status": "أقدر أساعدك بالتحقق من حالة الشبكة.",
        "network_complaint": "يمكننا تسجيل شكوى بخصوص مشكلة الشبكة.",
        "check_data_usage": "يمكنك معرفة استهلاك البيانات من التطبيق.",
        "balance_transfer": "يمكنك استخدام خدمة تحويل الرصيد.",
        "goodbye": "شكرًا لك، أنا موجود إذا احتجتِ أي مساعدة.",
        "feedback": "شكرًا لملاحظتك.",
        "thanks": "على الرحب والسعة.",
        "other": "لم أفهم الطلب بوضوح، ممكن توضحي أكثر؟",
    }

    english_responses = {
        "greeting": "Hello, how can I help you?",
        "slow_internet": "Your internet seems slow. Try restarting the router first.",
        "no_signal": "There seems to be a signal issue. Please restart your device and check coverage.",
        "payment_issue": "It looks like there is a payment issue. Please check your payment details and try again.",
        "renew_package": "I can help you renew your package. Which package do you want?",
        "offer_inquiry": "I can help you check the available offers.",
        "technical_support": "Sure, tell me the technical issue in detail.",
        "network_status": "I can help you check the network status.",
        "network_complaint": "We can register a network complaint.",
        "check_data_usage": "You can check your data usage from the app.",
        "balance_transfer": "You can use the balance transfer service.",
        "goodbye": "Thank you. I am here if you need help.",
        "feedback": "Thank you for your feedback.",
        "thanks": "You are welcome.",
        "other": "I did not understand clearly. Could you explain more?",
    }

    return (english_responses if lang == "en" else arabic_responses).get(intent, arabic_responses["other"])


def process_message(user_text: str, metrics: dict | None = None) -> dict:
    lang = detect_language(user_text)

    if intent_ready and predict_intent is not None:
        intent = predict_intent(user_text, lang)
    else:
        intent = "other"

    if sentiment_ready and predict_sentiment is not None:
        sentiment = predict_sentiment(user_text, lang)
    else:
        sentiment = "neutral"

    if prediction_ready and predict_network_issue is not None and metrics:
        prediction = predict_network_issue(metrics)
    else:
        prediction = None

    decision = None
    if prediction == 1 and intent in ["slow_internet", "no_signal", "network_complaint", "network_status"]:
        decision = {"rule_used": "technical_high_risk"}

    if response_ready and build_final_response is not None:
        response = build_final_response(
            intent=intent,
            sentiment=sentiment,
            decision=decision,
            lang=lang
        )
    else:
        response = fallback_response(intent, lang)

    return {
        "language": lang,
        "intent": intent,
        "sentiment": sentiment,
        "prediction": prediction,
        "response": response,
    }


# =========================
# Sidebar
# =========================
with st.sidebar:
    st.header("حالة الملفات")
    st.write("Intent model:", "جاهز" if intent_ready else "غير جاهز")
    st.write("Sentiment model:", "جاهز" if sentiment_ready else "غير جاهز")
    st.write("Prediction model:", "جاهز" if prediction_ready else "غير جاهز")
    st.write("Response logic:", "جاهز" if response_ready else "غير جاهز")

    with st.expander("تفاصيل الأخطاء"):
        if intent_error:
            st.code(intent_error)
        if sentiment_error:
            st.code(sentiment_error)
        if prediction_error:
            st.code(prediction_error)
        if response_error:
            st.code(response_error)


# =========================
# Main UI
# =========================
st.subheader("تجربة الشات بوت")

user_text = st.text_area("اكتبي رسالة العميل هنا", height=120)

use_metrics = st.checkbox("إضافة بيانات الشبكة للتنبؤ بالمشكلة")

metrics = None
if use_metrics:
    col1, col2 = st.columns(2)
    with col1:
        latency = st.number_input("Latency", min_value=0.0, value=0.0)
        packet_loss = st.number_input("Packet loss", min_value=0.0, value=0.0)
    with col2:
        signal_strength = st.number_input("Signal strength", value=0.0)
        connected_users = st.number_input("Connected users", min_value=0, value=0)

    metrics = {
        "latency": latency,
        "packet_loss": packet_loss,
        "signal_strength": signal_strength,
        "connected_users": connected_users,
    }

if st.button("إرسال"):
    if not user_text.strip():
        st.warning("اكتبي رسالة أولاً")
    else:
        result = process_message(user_text, metrics)

        st.success("تم تحليل الرسالة")

        st.markdown("### الرد")
        st.write(result["response"])

        st.markdown("### نتائج التحليل")
        st.write("اللغة:", result["language"])
        st.write("Intent:", result["intent"])
        st.write("Sentiment:", result["sentiment"])
        st.write("Network prediction:", result["prediction"])


2026-04-25 13:24:58.629 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 13:24:58.630 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 13:24:58.750 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-25 13:24:58.751 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 13:24:58.752 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 13:24:58.754 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 13:24:58.755 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn